# 🐄 Bengal Cattle 5Y Dataset — EDA
**Source:** `s3://cid-mbs/cow-data-v2/cow_master_dataset.csv`  
**Coverage:** 2021 – 2025 &nbsp;|&nbsp; 2,272 cattle records  
**Requires:** `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` set in `.env`

In [ ]:
# Install dependencies (safe to re-run)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'boto3', 'pandas', 'matplotlib', 'seaborn'], check=True)
print('✅ Dependencies ready')

In [ ]:
import io, os, re, warnings
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv
load_dotenv()
warnings.filterwarnings('ignore')

# ── AWS credentials (loaded from .env) ───────────────────────────────────────
AWS_KEY    = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET = os.getenv('AWS_SECRET_ACCESS_KEY')
REGION     = 'us-east-1'
BUCKET     = 'cid-mbs'
S3_KEY     = 'cow-data-v2/cow_master_dataset.csv'

# ── Load from S3 ──────────────────────────────────────────────────────────────
session = boto3.Session(aws_access_key_id=AWS_KEY,
                        aws_secret_access_key=AWS_SECRET,
                        region_name=REGION)
obj = session.client('s3').get_object(Bucket=BUCKET, Key=S3_KEY)
df  = pd.read_csv(io.BytesIO(obj['Body'].read()), dtype={'year': str})

# ── Pure numeric columns ──────────────────────────────────────────────────────
num_cols = ['weight_in_kg', 'price', 'height_in_inch', 'sale_offer_percentage',
            'teeth', 'parts_available']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# ── Age & feedlot stored as '2 Years' / '3 Months' — parse to numeric months ─
def to_months(val):
    if pd.isna(val): return np.nan
    m = re.match(r'([\d.]+)\s*(year|month)', str(val).lower().strip())
    if not m: return np.nan
    n, unit = float(m.group(1)), m.group(2)
    return round(n * 12 if 'year' in unit else n, 1)

df['age_months']     = df['age_in_month'].apply(to_months)
df['feedlot_months'] = df['feedlot_in_month'].apply(to_months)

# ── Boolean columns ───────────────────────────────────────────────────────────
bool_cols = ['is_dewormed', 'is_fmd_vaccinated', 'is_anthrax_vaccinated',
             'is_lumpy_skin_disease', 'is_special']
for c in bool_cols:
    df[c] = df[c].map({'True': True, 'False': False, True: True, False: False})

df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce')

print(f'✅  Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head(3)

## 1 · Dataset Overview

In [ ]:
print('Shape :', df.shape)
print('\nMissing values (top 10):')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nNumerical summary:')
df[num_cols + ['age_months', 'feedlot_months']].describe().round(2)

## 2 · Records per Year

In [ ]:
PALETTE = sns.color_palette('Set2', 5)
sns.set_theme(style='whitegrid', font_scale=1.1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

year_counts = df['year'].value_counts().sort_index()
axes[0].bar(year_counts.index, year_counts.values, color=PALETTE, edgecolor='white', linewidth=1.2)
for i, (yr, v) in enumerate(year_counts.items()):
    axes[0].text(i, v + 8, str(v), ha='center', fontsize=10, fontweight='bold')
axes[0].set(title='Cattle Records per Year', xlabel='Year', ylabel='Count')
axes[0].set_xticks(range(len(year_counts)))
axes[0].set_xticklabels(year_counts.index)

axes[1].pie(year_counts.values, labels=year_counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Year Share')

plt.tight_layout(); plt.show()

## 3 · Weight Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['weight_in_kg'].dropna(), kde=True, bins=40,
             color='steelblue', ax=axes[0])
axes[0].axvline(df['weight_in_kg'].median(), color='tomato', linestyle='--',
                label=f"Median {df['weight_in_kg'].median():.0f} kg")
axes[0].set(title='Weight Distribution (all years)', xlabel='Weight (kg)', ylabel='Count')
axes[0].legend()

sns.boxplot(data=df, x='year', y='weight_in_kg', palette='Set2', ax=axes[1])
axes[1].set(title='Weight by Year', xlabel='Year', ylabel='Weight (kg)')

plt.tight_layout(); plt.show()
df.groupby('year')['weight_in_kg'].agg(['mean','median','min','max','std']).round(1)

## 4 · Breed & Sex Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

breed_counts = df['breed'].value_counts().head(12)
sns.barplot(x=breed_counts.values, y=breed_counts.index, palette='viridis', ax=axes[0])
axes[0].set(title='Top 12 Breeds', xlabel='Count', ylabel='')
for i, v in enumerate(breed_counts.values):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=9)

sex_year = df.groupby(['year', 'sex']).size().unstack(fill_value=0)
sex_year.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set(title='Sex Distribution by Year', xlabel='Year', ylabel='Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sex', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout(); plt.show()

## 5 · Price Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

sns.histplot(df['price'].dropna(), bins=50, kde=True, color='mediumpurple', ax=axes[0])
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[0].set(title='Price Distribution', xlabel='Price (BDT)', ylabel='Count')

for yr, grp in df.dropna(subset=['weight_in_kg','price']).groupby('year'):
    axes[1].scatter(grp['weight_in_kg'], grp['price']/1000, label=yr, alpha=0.5, s=18)
axes[1].set(title='Price vs Weight', xlabel='Weight (kg)', ylabel='Price (BDT 000s)')
axes[1].legend(title='Year', fontsize=8)

med_price = df.groupby('year')['price'].median()
axes[2].plot(med_price.index, med_price.values / 1000, marker='o',
             color='darkorange', linewidth=2, markersize=8)
for yr, val in med_price.items():
    axes[2].annotate(f'{val/1000:.0f}k', (yr, val/1000),
                     textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=9, fontweight='bold')
axes[2].set(title='Median Price Trend', xlabel='Year', ylabel='Median Price (BDT 000s)')

plt.tight_layout(); plt.show()

## 6 · Vaccination & Health Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

vacc_labels = ['Dewormed', 'FMD Vaccinated', 'Anthrax Vaccinated', 'Lumpy Skin Disease']
vacc_cols   = bool_cols[:4]
vacc_rates  = [df[c].mean() * 100 for c in vacc_cols]
bars = axes[0].barh(vacc_labels, vacc_rates, color=sns.color_palette('RdYlGn', 4))
for bar, rate in zip(bars, vacc_rates):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{rate:.1f}%', va='center', fontsize=10, fontweight='bold')
axes[0].set(title='Health & Vaccination Rates (%)', xlim=(0, 108), xlabel='% of cattle')

vacc_year = df.groupby('year')[vacc_cols].mean() * 100
vacc_year.T.plot(kind='bar', ax=axes[1], colormap='Set1', edgecolor='white')
axes[1].set(title='Vaccination Rates by Year (%)', ylabel='%')
axes[1].set_xticklabels(vacc_labels, rotation=20, ha='right')
axes[1].legend(title='Year', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

plt.tight_layout(); plt.show()

## 7 · Size & Color Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

size_order = [s for s in ['MINIMUM','SMALL','MEDIUM','LARGE','EXTRA_LARGE']
              if s in df['size'].unique()]
size_year  = df.groupby(['year','size']).size().unstack(fill_value=0)
size_year[size_order].plot(kind='bar', stacked=True, ax=axes[0],
                           colormap='Spectral', edgecolor='white')
axes[0].set(title='Size Category by Year', xlabel='Year', ylabel='Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Size', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

color_counts = df['color'].value_counts()
axes[1].pie(color_counts.values, labels=color_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', len(color_counts)),
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Coat Color Distribution')

plt.tight_layout(); plt.show()

## 8 · Listing Status & Availability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

status_counts = df['status'].value_counts()
sns.barplot(x=status_counts.values, y=status_counts.index, palette='coolwarm', ax=axes[0])
axes[0].set(title='Listing Status', xlabel='Count', ylabel='')
for i, v in enumerate(status_counts.values):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=9)

status_year = df.groupby(['year','status']).size().unstack(fill_value=0)
status_year.plot(kind='bar', ax=axes[1], colormap='tab10', edgecolor='white')
axes[1].set(title='Status by Year', xlabel='Year', ylabel='Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Status', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)

plt.tight_layout(); plt.show()

## 9 · Weight vs Age — Scatter by Breed (Top 6)

In [ ]:
top6_breeds = df['breed'].value_counts().head(6).index
plot_df = df[df['breed'].isin(top6_breeds)].dropna(subset=['age_months','weight_in_kg'])

fig, ax = plt.subplots(figsize=(10, 5))
for breed, grp in plot_df.groupby('breed'):
    ax.scatter(grp['age_months'], grp['weight_in_kg'], label=breed, alpha=0.55, s=20)
ax.set(title='Weight vs Age (Top 6 Breeds)', xlabel='Age (months)', ylabel='Weight (kg)')
ax.legend(title='Breed', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

## 10 · Correlation Heatmap

In [ ]:
corr_cols = ['weight_in_kg', 'price', 'height_in_inch', 'age_months',
             'feedlot_months', 'sale_offer_percentage', 'teeth', 'parts_available']
corr_df = df[corr_cols].corr()

mask = np.zeros_like(corr_df, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, mask=mask, annot_kws={'size': 9})
ax.set_title('Numeric Feature Correlations')
plt.tight_layout(); plt.show()

## 11 · Quick Summary Stats

In [ ]:
has_video = df['youtube_slug'].notna() & (df['youtube_slug'].str.strip() != '')
fully_vacc = df[bool_cols[:3]].all(axis=1)

summary = {
    'Total cattle'         : len(df),
    'Years covered'        : ', '.join(sorted(df['year'].unique())),
    'Unique breeds'        : df['breed'].nunique(),
    'Avg weight (kg)'      : round(df['weight_in_kg'].mean(), 1),
    'Median price (BDT)'   : f"{df['price'].median():,.0f}",
    'Heaviest cow (kg)'    : df['weight_in_kg'].max(),
    'Avg age (months)'     : round(df['age_months'].mean(), 1),
    '% with YouTube video' : f"{has_video.mean()*100:.1f}%",
    '% fully vaccinated'   : f"{fully_vacc.mean()*100:.1f}%",
    'Most common breed'    : df['breed'].mode()[0],
    'Most common status'   : df['status'].mode()[0],
}

print('=' * 46)
print('        DATASET SUMMARY')
print('=' * 46)
for k, v in summary.items():
    print(f'  {k:<28} {v}')
print('=' * 46)